# 02_bag_of_words_tfidf: Vector Representations from Scratch

This notebook builds Bag of Words (BoW) and Term Frequency - Inverse Document Frequency (TF-IDF) representation matrices from scratch using NumPy, and compares results against Scikit-Learn's estimators.


In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Tiny Corpus
corpus = [
    "cat feline mat",
    "feline rug mat"
]

# 1. Map vocabulary
vocab = sorted(list(set(" ".join(corpus).split())))
word_to_idx = {w: i for i, w in enumerate(vocab)}
print("Vocabulary mapping:", word_to_idx)

# 2. Bag of Words (BoW) from scratch
bow_matrix = np.zeros((len(corpus), len(vocab)))
for doc_idx, doc in enumerate(corpus):
    for word in doc.split():
        if word in word_to_idx:
            bow_matrix[doc_idx, word_to_idx[word]] += 1

print("\nBag of Words Matrix (from scratch):\n", bow_matrix)

# 3. Smooth TF-IDF from scratch
# Smooth IDF formulation: log((1 + N) / (1 + DF)) + 1
N = len(corpus)
df = np.sum(bow_matrix > 0, axis=0)
idf = np.log((1 + N) / (1 + df)) + 1

# Calculate TF-IDF
tfidf_matrix = bow_matrix * idf

# L2 normalization to match Scikit-Learn standard
norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
tfidf_norm = tfidf_matrix / norms

print("\nTF-IDF Matrix (from scratch, normalized):\n", tfidf_norm)

# 4. Compare with Scikit-Learn
vectorizer = TfidfVectorizer(norm='l2', smooth_idf=True, use_idf=True)
sklearn_tfidf = vectorizer.fit_transform(corpus).toarray()
print("\nScikit-Learn TF-IDF Matrix:\n", sklearn_tfidf)

# Check assertion
assert np.allclose(tfidf_norm, sklearn_tfidf, atol=1e-5)
print("\nSUCCESS: Custom TF-IDF matrix matches Scikit-Learn output exactly!")


Vocabulary mapping: {'cat': 0, 'feline': 1, 'mat': 2, 'rug': 3}

Bag of Words Matrix (from scratch):
 [[1. 1. 1. 0.]
 [0. 1. 1. 1.]]

TF-IDF Matrix (from scratch, normalized):
 [[0.70490949 0.50154891 0.50154891 0.        ]
 [0.         0.50154891 0.50154891 0.70490949]]

Scikit-Learn TF-IDF Matrix:
 [[0.70490949 0.50154891 0.50154891 0.        ]
 [0.         0.50154891 0.50154891 0.70490949]]

SUCCESS: Custom TF-IDF matrix matches Scikit-Learn output exactly!


### Output Explanation
- The hand-coded matrix implementation calculates raw term frequencies, applies the smooth IDF equation, normalizes vectors by their $L_2$ Euclidean norms, and matches the output of Scikit-Learn's `TfidfVectorizer` exactly.
